# 6 · Human-in-the-loop

A node can pause the flow to request human input. The engine:

1. Catches `HumanInputRequired` from the node
2. Checkpoints state to a JSON-serializable dict (safe to store in a DB)
3. Emits a `flow_paused` event (or raises `FlowPausedException` in sync mode)

Later — seconds or days later, possibly in a different process — call `resume()` / `resume_sync()` with the checkpoint and the human's response. The response becomes the paused node's output and flows downstream.

**FlowStore state survives the checkpoint/resume cycle.**

In [ ]:
import json
from typing import Annotated

from conductor import GraphEdge, GraphNode, NodeRegistry, compile
from conductor.errors import FlowPausedException, HumanInputRequired
from conductor.execution.engine import collect, execute, resume
from conductor.widgets import Output, Text, Textarea

registry = NodeRegistry()


@registry.node("draft", version=1, name="Draft", description="Drafts a message")
def draft(
    topic: Annotated[str, Text(label="Topic")],
) -> Annotated[str, Output(label="Draft")]:
    return (
        f"Dear team,\n\n"
        f"Re: {topic}\n\n"
        f"This is the auto-generated draft about {topic}.\n\n"
        "Best regards"
    )


@registry.node("send", version=1, name="Send", description="Sends the message")
def send(
    message: Annotated[str, Textarea(label="Message")],
) -> Annotated[str, Output(label="Status")]:
    print(f"  [SEND] would send:\n    {message[:80]}...")
    return "sent" 

## The approval gate

A node pauses by raising `HumanInputRequired`. The engine catches this — it's a signal, not an error.

In [ ]:
@registry.node("approve", version=1, name="Approval Gate", description="Requires human approval")
def approve(
    text: Annotated[str, Textarea(label="Content to review")],
) -> Annotated[str, Output(label="Approved content")]:
    raise HumanInputRequired(
        prompt=f"Please review and approve this content:\n\n{text[:200]}",
        schema={
            "approved": "bool",
            "edited_text": "str (optional — leave empty to use original)",
        },
    )

## Build the flow

```text
  draft ──> approve ──> send
```

In [ ]:
nodes = [
    GraphNode("n1", "draft@1", {"topic": "Q3 roadmap"}),
    GraphNode("n2", "approve@1", None),
    GraphNode("n3", "send@1", None),
]
edges = [
    GraphEdge("e1", "n1", "n2", "result", "text"),
    GraphEdge("e2", "n2", "n3", "result", "message"),
]

compiled = compile(nodes=nodes, edges=edges, registry=registry)

## Phase 1 — run until pause

`collect()` raises `FlowPausedException` when it encounters a `flow_paused` event. The exception carries the checkpoint dict.

(The sync-script equivalent is `execute_sync(compiled)`, which wraps this with `asyncio.run`.)

In [ ]:
try:
    await collect(execute(compiled))
    print("ERROR: should have paused")
except FlowPausedException as e:
    checkpoint = e.checkpoint
    print(f"Flow paused at node: {checkpoint['waiting_node_id']}")
    print(f"Prompt:\n{checkpoint['prompt']}\n")
    print(f"Expected response schema: {checkpoint['input_schema']}")

# The checkpoint is a plain dict — JSON-serializable
checkpoint_json = json.dumps(checkpoint, indent=2)
print(f"\nCheckpoint size: {len(checkpoint_json)} bytes (store this in your DB)")

## Phase 2 — resume with the human's response

The response becomes the output of the paused node. It flows through the edge to `send`.

In production: load the checkpoint and flow definition from your database (possibly much later), re-compile the graph, and call `resume()` (or `resume_sync()` from a plain script).

In [ ]:
human_response = "APPROVED: Dear team, Re: Q3 roadmap — looks great, ship it!"

restored_checkpoint = json.loads(checkpoint_json)
results = await collect(resume(compiled, restored_checkpoint, response=human_response))

print("\n=== Final results ===")
for node_id, result in results.items():
    print(f"  {node_id}: {result}")